# Harm prediction via 11 event-type buckets: main model + ablation (Colab)

Trains the main model and its ablation on Colab:

- **Shared encoder:** `emilyalsentzer/Bio_ClinicalBERT`, fine-tuned (standard BERT-base, 512-token limit). Its CLS token is the shared representation.
  It replaced `yikuan8/Clinical-Longformer`, which padded every input to its 512-token attention window even though the narratives are short (median ~76 tokens), so its long-document machinery was pure overhead.
- **Head A:** linear layer → 11-way softmax (event type), cross-entropy.
- **Head B:** linear layer on **[encoder output ; Head A's 11-way soft probabilities]** → sigmoid (hurt), BCE. Soft probabilities, not an argmax, so gradients flow end to end.
- **Loss:** `w_a * CE + w_b * BCE`, masked per row (the 64 uncoded rows train Head B only; rows with no harm score train Head A only).
- **Ablation:** the same model, but Head B sees only the encoder output.
- **Threshold:** swept on **validation** for ≥95% hurt recall, then applied to test.
- **Training:** validation early stopping using the same score as the baseline (val harm PR-AUC + event macro-F1), checked every `eval_every` steps with a `patience` limit and a `max_epochs` cap; the best weights are restored.
- **Speed:** fp16 mixed precision; dynamic per-batch padding with length-bucketed batches; `max_length=320` (covers 99.8% of inputs); the largest batch size that fits the GPU (probed in step 5b) at the same 32 rows per optimizer step. Step 5c times ~50 steps so you can check the ETA before committing.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `LOCAL_ONLY_student_facing_candidate (1).csv` to `MyDrive/psrs_harm/` (or change `DRIVE_DIR` below).

Everything (processed data, checkpoints, predictions, report) is written under `DRIVE_DIR` on Google Drive, never only to `/content`, which Colab wipes on disconnect. If the session disconnects, re-run all cells: training resumes from the latest checkpoint.

## 1. Mount Google Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU: switch runtime to T4')

## 2. Configuration

In [ ]:
import os
DRIVE_DIR = '/content/drive/MyDrive/psrs_harm'          # all persistent state lives here
CSV_PATH = os.path.join(DRIVE_DIR, 'LOCAL_ONLY_student_facing_candidate (1).csv')
OUT_DIR = os.path.join(DRIVE_DIR, 'outputs')

# Guardrail: refuse to checkpoint anywhere Colab will wipe.
assert OUT_DIR.startswith('/content/drive/'), 'Checkpoints must go to Google Drive, not Colab local disk'
assert os.path.exists(CSV_PATH), f'Upload the CSV to {CSV_PATH}'
os.makedirs(OUT_DIR, exist_ok=True)

EFFECTIVE_BATCH = 32       # rows per optimizer step, same as the original 8 x 4, so optimization dynamics are unchanged

BASE_CFG = dict(
    model_name='emilyalsentzer/Bio_ClinicalBERT',
    max_length=320,        # covers 99.83% of inputs untruncated (p99 = 281 tokens after tokenization; 133 of 80k truncated vs 7 at 512)
    batch_size=8,          # placeholder: step 5b replaces this with the largest batch that fits the GPU...
    grad_accum=4,          # ...and sets grad_accum = EFFECTIVE_BATCH // batch_size
    length_bucketing=True, # similar-length reports batched together; each batch padded only to its longest report
    eval_batch_size=128,
    lr=2e-5,
    max_epochs=4,          # cap only: validation early stopping normally ends sooner
    eval_every=500,        # optimizer steps between validation checks (~3.75 checks per epoch)
    patience=3,            # stop after 3 checks without improvement (same score as the baseline's early stopping)
    w_a=1.0, w_b=1.0,      # start equal; adjust if one head stalls (loss-imbalance warning)
    seed=42,
    log_every=50,          # the step-50 log line prints steps/s and ETA
    ckpt_every=250,        # optimizer steps between Drive checkpoints
    keep_ckpts=2,
    fp16=True,             # mixed precision on CUDA (autocast fp16 + GradScaler)
)
# One checkpoint folder per encoder, so switching models never resumes from another model's checkpoints.
CKPT_ROOT = os.path.join(OUT_DIR, 'checkpoints', BASE_CFG['model_name'].split('/')[-1])
CONFIGS = {
    'main':          dict(BASE_CFG, use_event_probs=True,  ckpt_dir=os.path.join(CKPT_ROOT, 'main')),
    'main_ablation': dict(BASE_CFG, use_event_probs=False, ckpt_dir=os.path.join(CKPT_ROOT, 'main_ablation')),
}

## 3. Install dependencies

In [ ]:
!pip -q install transformers scikit-learn pandas pyarrow matplotlib

## 4. Project code

These cells are generated from `src/` by `src/build_notebook.py`, so they are the same code the local smoke test ran.

In [ ]:
%%writefile pipeline.py
"""Data pipeline for the harm / event-type model.

This file is the source of truth for field names, the harm cut point,
the 11-bucket rule and the exclusion list. Every check here raises (hard-fails)
rather than logging, and uses explicit `raise` so it survives `python -O`.
"""
import re

import numpy as np
import pandas as pd

CSV_NAME = "LOCAL_ONLY_student_facing_candidate (1).csv"

ID_COL = "Event No."
DATE_COL = "Event Date"
EVENT_TYPE_COL = "Event Type"
HARM_COL = "Significance (PSRS Harm score)"
TEXT_COL = "event_comments"

# Post-investigation fields: must never reach the model.
EXCLUDED_COLUMNS = (
    "manager_comments",
    "unit_actions_taken",
    "shareable_lessons",
    "HPI Designation... Name",
    "Analyst-Report Type*",
    "Level of Invet",
)
# Label sources are also forbidden as features (they are the answers).
LABEL_SOURCE_COLUMNS = (EVENT_TYPE_COL, HARM_COL)

# Intake fields used as model input: narrative plus a short structured prefix.
PREFIX_FIELDS = (
    ("Unit", ("Location Name",)),
    ("Service", ("Encounter Service",)),
    ("Age", ("Age at Encounter",)),
    ("Prescribed", ("ME - Prescribed - Name *... Name", "ME - Prescribed - Dose *")),
    ("Administered", ("ME - Admin - Name", "ME - Admin - Dose")),
    ("ADR suspect medication", ("ADR - Suspect Med Name", "ADR - Dose *")),
)
FEATURE_COLUMNS = (TEXT_COL,) + tuple(c for _, cols in PREFIX_FIELDS for c in cols)

# Index in this tuple = class id for event_type_label.
EVENT_BUCKETS = ("ADR", "C", "E", "EQ", "FALL", "I", "ME", "O", "SH", "SI", "T")
N_EVENT_CLASSES = len(EVENT_BUCKETS)

SPLITS = ("TRAIN", "VALIDATION", "TEST")
SPLIT_MONTHS = {"TRAIN": set(range(1, 9)), "VALIDATION": {9}, "TEST": {10}}
# Audit: 15.11% / 4.13% / 4.85% hurt among scored rows (split / distribution-shift guardrail).
EXPECTED_HURT_PREVALENCE = {
    "TRAIN": (0.135, 0.165),
    "VALIDATION": (0.030, 0.055),
    "TEST": (0.035, 0.065),
}

UNLABELED = -1


class DataContractError(RuntimeError):
    """Raised when the data violates a data-contract guarantee."""


class LeakageError(DataContractError):
    """Raised when an excluded or label-source column reaches the feature set."""


def assert_no_excluded_columns(columns):
    """Hard-fail if any excluded or label-source column is in `columns`."""
    leaked = sorted(set(columns) & (set(EXCLUDED_COLUMNS) | set(LABEL_SOURCE_COLUMNS)))
    if leaked:
        raise LeakageError(f"Excluded/label columns in model features: {leaked}")


def event_bucket(event_type):
    """Code before the first '-', uppercased. None if there is no '-'."""
    if not isinstance(event_type, str) or "-" not in event_type:
        return None
    return event_type.split("-", 1)[0].strip().upper()


def harm_label(score):
    """A-D -> 0, E-I -> 1, blank -> UNLABELED."""
    if not isinstance(score, str) or not score.strip():
        return UNLABELED
    letter = score.strip()[0].upper()
    if letter in "ABCD":
        return 0
    if letter in "EFGHI":
        return 1
    raise DataContractError(f"Unexpected harm score: {score!r}")


def split_from_event_no(event_no):
    m = re.match(r"^SYNPROD-(TRAIN|VALIDATION|TEST)-\d+$", str(event_no))
    if not m:
        raise DataContractError(f"Event No. without a split prefix: {event_no!r}")
    return m.group(1)


def build_input_text(features):
    """'Unit: X. Service: Y. ...' prefix + narrative. Event type is never included."""
    assert_no_excluded_columns(features.columns)
    missing = set(FEATURE_COLUMNS) - set(features.columns)
    if missing:
        raise DataContractError(f"Feature columns missing: {sorted(missing)}")

    def fmt(v):
        if isinstance(v, float) and v.is_integer():
            return str(int(v))
        return str(v).strip()

    parts = []
    for label, cols in PREFIX_FIELDS:
        vals = features[list(cols)]
        joined = vals.apply(lambda r: " ".join(fmt(v) for v in r if pd.notna(v) and str(v).strip()), axis=1)
        parts.append(np.where(joined != "", label + ": " + joined + ". ", ""))
    prefix = pd.Series(["".join(p) for p in zip(*parts)], index=features.index)
    return (prefix.str.strip() + "\n" + features[TEXT_COL].astype(str)).str.lstrip()


def load_dataset(csv_path, nrows=None):
    """Load the CSV and return one row per report with labels and model input text.

    Columns: event_no, split, event_bucket, event_type_label, harm_label, text.
    Excluded columns are never read from disk.
    """
    usecols = [ID_COL, DATE_COL, EVENT_TYPE_COL, HARM_COL, *FEATURE_COLUMNS]
    raw = pd.read_csv(csv_path, usecols=usecols, nrows=nrows, low_memory=False)

    split = raw[ID_COL].map(split_from_event_no)
    month = pd.to_datetime(raw[DATE_COL], errors="raise").dt.month
    bad = [(s, m) for s, m in zip(split, month) if m not in SPLIT_MONTHS[s]]
    if bad:
        raise DataContractError(f"{len(bad)} rows where Event No. split disagrees with Event Date, e.g. {bad[:3]}")

    bucket = raw[EVENT_TYPE_COL].map(event_bucket)
    unknown = sorted(set(bucket.dropna()) - set(EVENT_BUCKETS))
    if unknown:
        raise DataContractError(f"Event-type codes outside the 11 buckets: {unknown}")

    features = raw[list(FEATURE_COLUMNS)]
    assert_no_excluded_columns(features.columns)

    out = pd.DataFrame({
        "event_no": raw[ID_COL],
        "split": split,
        "event_bucket": bucket,
        "event_type_label": bucket.map({b: i for i, b in enumerate(EVENT_BUCKETS)}).fillna(UNLABELED).astype(int),
        "harm_label": raw[HARM_COL].map(harm_label).astype(int),
        "text": build_input_text(features),
    })
    return out


def check_head_a_classes(df, name):
    """Head A sets must contain exactly the 11 buckets, no more, no fewer."""
    labels = df.loc[df.event_type_label != UNLABELED, "event_type_label"].unique()
    if len(labels) != N_EVENT_CLASSES or set(labels) != set(range(N_EVENT_CLASSES)):
        present = sorted(EVENT_BUCKETS[i] if 0 <= i < N_EVENT_CLASSES else f"<unknown id {i}>" for i in labels)
        raise DataContractError(f"{name}: expected exactly {N_EVENT_CLASSES} event-type labels, got {len(labels)}: {present}")


def check_hurt_prevalence(df):
    """Guardrail: a broken split shows up as wrong prevalence."""
    report = {}
    for s in SPLITS:
        y = df.loc[(df.split == s) & (df.harm_label != UNLABELED), "harm_label"]
        prev = float(y.mean())
        lo, hi = EXPECTED_HURT_PREVALENCE[s]
        if not lo <= prev <= hi:
            raise DataContractError(f"{s} hurt prevalence {prev:.4f} outside expected [{lo}, {hi}]")
        report[s] = prev
    return report


def validate_full_dataset(df):
    """All hard checks for the full 80k file. Returns a summary dict."""
    counts = df.split.value_counts().to_dict()
    if counts != {"TRAIN": 60000, "VALIDATION": 10000, "TEST": 10000}:
        raise DataContractError(f"Unexpected split sizes: {counts}")
    for s in SPLITS:
        check_head_a_classes(df[df.split == s], s)
    prevalence = check_hurt_prevalence(df)
    rare = {EVENT_BUCKETS[i]: int(n) for i, n in
            df[(df.split == "TRAIN") & (df.event_type_label >= 0)].event_type_label.value_counts().items() if n < 50}
    return {
        "split_sizes": counts,
        "hurt_prevalence": prevalence,
        "no_event_code_rows": int((df.event_type_label == UNLABELED).sum()),
        "no_harm_score_rows": int((df.harm_label == UNLABELED).sum()),
        "rare_train_buckets_lt50": rare,
    }


def smoke_slice(df, n=500, min_per_bucket=3, min_hurt=40, seed=0):
    """Small TRAIN slice with every bucket and enough hurt cases for the smoke test."""
    rng = np.random.RandomState(seed)
    tr = df[df.split == "TRAIN"]
    picks = set()
    for i in range(N_EVENT_CLASSES):
        idx = tr.index[tr.event_type_label == i]
        picks.update(rng.choice(idx, min(min_per_bucket, len(idx)), replace=False))
    hurt = tr.index[tr.harm_label == 1].difference(list(picks))
    picks.update(rng.choice(hurt, min_hurt, replace=False))
    picks.update(rng.choice(tr.index[tr.event_type_label == UNLABELED], 2, replace=False))
    rest = tr.index.difference(list(picks))
    picks.update(rng.choice(rest, n - len(picks), replace=False))
    return df.loc[sorted(picks)]

In [ ]:
%%writefile mtl.py
"""Multi-task model pieces shared by the baseline, the main model and the ablation.

Head A: shared representation -> 11-way event-type logits.
Head B: [shared representation ; softmax(Head A)] -> 1 hurt logit. The soft probability
vector (not an argmax) is concatenated, and gradients flow through it end to end.
Ablation: Head B sees the shared representation only (use_event_probs=False).
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from pipeline import N_EVENT_CLASSES, UNLABELED


def _head(in_dim, out_dim, hidden, dropout):
    if hidden:
        return nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, hidden), nn.ReLU(),
                             nn.Dropout(dropout), nn.Linear(hidden, out_dim))
    return nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, out_dim))


class MultiTaskHeads(nn.Module):
    def __init__(self, in_dim, use_event_probs=True, hidden=0, dropout=0.1):
        super().__init__()
        self.use_event_probs = use_event_probs
        self.head_a = _head(in_dim, N_EVENT_CLASSES, hidden, dropout)
        self.head_b = _head(in_dim + (N_EVENT_CLASSES if use_event_probs else 0), 1, hidden, dropout)

    def forward(self, h, event_probs_override=None):
        logits_a = self.head_a(h)
        if self.use_event_probs:
            probs_a = F.softmax(logits_a, dim=-1) if event_probs_override is None else event_probs_override
            logit_b = self.head_b(torch.cat([h, probs_a], dim=-1)).squeeze(-1)
        else:
            logit_b = self.head_b(h).squeeze(-1)
        return logits_a, logit_b


class EmbeddingMultiTask(nn.Module):
    """Baseline: frozen sentence embeddings in, heads on top."""

    def __init__(self, in_dim, use_event_probs=True, hidden=256, dropout=0.2):
        super().__init__()
        self.heads = MultiTaskHeads(in_dim, use_event_probs, hidden, dropout)

    def forward(self, emb, event_probs_override=None):
        return self.heads(emb, event_probs_override)


class EncoderMultiTask(nn.Module):
    """Main model: fine-tuned transformer encoder (CLS token) + linear heads.

    If the encoder is a Longformer, the CLS token also gets global attention.
    """

    def __init__(self, model_name, use_event_probs=True, dropout=0.1):
        super().__init__()
        from transformers import AutoModel
        self.encoder = AutoModel.from_pretrained(model_name)
        self.is_longformer = "longformer" in self.encoder.config.model_type
        self.heads = MultiTaskHeads(self.encoder.config.hidden_size, use_event_probs, hidden=0, dropout=dropout)

    def forward(self, input_ids, attention_mask, event_probs_override=None):
        kwargs = {}
        if self.is_longformer:
            g = torch.zeros_like(input_ids)
            g[:, 0] = 1
            kwargs["global_attention_mask"] = g
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs).last_hidden_state[:, 0]
        return self.heads(h, event_probs_override)


def multitask_loss(logits_a, logit_b, y_event, y_harm, w_a=1.0, w_b=1.0):
    """w_a * CE(event) + w_b * BCE(harm), each over the rows that have that label."""
    zero = logit_b.sum() * 0.0
    ma = y_event != UNLABELED
    mb = y_harm != UNLABELED
    ce = F.cross_entropy(logits_a[ma], y_event[ma]) if ma.any() else zero
    bce = F.binary_cross_entropy_with_logits(logit_b[mb], y_harm[mb].float()) if mb.any() else zero
    return w_a * ce + w_b * bce, ce.detach(), bce.detach()


@torch.no_grad()
def predict(model, batches, device, event_prior=None):
    """Run the model over `batches` (iterable of (inputs_dict, y_event, y_harm)).

    Returns probs_a (N, 11), p_hurt (N,), and, if Head B uses Head A and `event_prior`
    is given, p_hurt_prior: Head B's output with Head A's vector replaced by the train
    prior -- used by the event-type-reliance guardrail to see whether Head B relies on Head A.
    """
    model.eval()
    probs_a, p_hurt, p_prior = [], [], []
    uses_a = model.heads.use_event_probs if hasattr(model, "heads") else False
    for inputs, _, _ in batches:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        logits_a, logit_b = model(**inputs)
        probs_a.append(F.softmax(logits_a, -1).float().cpu())
        p_hurt.append(torch.sigmoid(logit_b).float().cpu())
        if uses_a and event_prior is not None:
            prior = torch.as_tensor(event_prior, dtype=logits_a.dtype, device=device).expand_as(logits_a)
            _, lb = model(**inputs, event_probs_override=prior)
            p_prior.append(torch.sigmoid(lb).float().cpu())
    out = {"probs_a": torch.cat(probs_a).numpy(), "p_hurt": torch.cat(p_hurt).numpy()}
    if p_prior:
        out["p_hurt_prior"] = torch.cat(p_prior).numpy()
    return out


def linear_warmup_decay(optimizer, warmup_steps, total_steps):
    def f(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)


def event_prior_from_labels(y_event):
    y = np.asarray(y_event)
    y = y[y != UNLABELED]
    return np.bincount(y, minlength=N_EVENT_CLASSES) / len(y)


def seed_everything(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def pick_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

In [ ]:
%%writefile evaluation.py
"""Evaluation metrics, threshold selection, calibration and guardrails."""
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_fscore_support,
                             roc_auc_score)

from pipeline import EVENT_BUCKETS, N_EVENT_CLASSES, UNLABELED

TARGET_RECALL = 0.95
SH = EVENT_BUCKETS.index("SH")


class GuardrailWarning(UserWarning):
    pass


def warn(msg):
    warnings.warn(msg, GuardrailWarning, stacklevel=2)
    print(f"[GUARDRAIL WARNING] {msg}")


# ---------- Head B: threshold, metrics, calibration ----------

def choose_threshold(y_val, p_val, target_recall=TARGET_RECALL):
    """Highest threshold whose validation recall on the hurt class is >= target.

    Takes validation data only; never call this with test data.
    """
    y_val, p_val = np.asarray(y_val), np.asarray(p_val)
    order = np.argsort(-p_val)
    tp = np.cumsum(y_val[order])
    recall = tp / y_val.sum()
    k = int(np.argmax(recall >= target_recall))  # first (highest-score) cut reaching target
    thr = float(p_val[order][k])
    pred = p_val >= thr
    return {
        "threshold": thr,
        "val_recall": float(y_val[pred].sum() / y_val.sum()),
        "val_precision": float(y_val[pred].mean()),
        "target_recall": target_recall,
        "rule": f"highest threshold with validation hurt-class recall >= {target_recall:.0%}",
    }


def harm_metrics(y, p, threshold):
    y, p = np.asarray(y), np.asarray(p)
    pred = (p >= threshold).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "n": int(len(y)), "n_hurt": int(y.sum()), "prevalence": float(y.mean()),
        "threshold": float(threshold),
        "recall": float(rec), "precision": float(prec), "f1": float(f1),
        "pr_auc": float(average_precision_score(y, p)), "roc_auc": float(roc_auc_score(y, p)),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "flagged_frac": float(pred.mean()),
        "accuracy": float((pred == y).mean()),
        "always_not_hurt_accuracy": float(1 - y.mean()),  # the accuracy trap
    }


def calibration_table(y, p, n_bins=10):
    y, p = np.asarray(y), np.asarray(p)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        m = idx == b
        rows.append({"bin": f"{edges[b]:.0%}-{edges[b + 1]:.0%}", "n": int(m.sum()),
                     "mean_predicted": float(p[m].mean()) if m.any() else np.nan,
                     "observed_hurt_rate": float(y[m].mean()) if m.any() else np.nan})
    t = pd.DataFrame(rows)
    ece = float(np.nansum(t.n * (t.mean_predicted - t.observed_hurt_rate).abs()) / len(y))
    return t, ece


class PlattCalibrator:
    """Logistic recalibration of the hurt logit, fit on validation.

    Monotonic, so it does not change the ranking or the recall/precision achieved at
    the validation-chosen operating point; it only makes the probabilities honest.
    """

    def fit(self, y_val, p_val):
        self.lr = LogisticRegression().fit(self._logit(p_val), y_val)
        return self

    def transform(self, p):
        return self.lr.predict_proba(self._logit(p))[:, 1]

    @staticmethod
    def _logit(p):
        p = np.clip(np.asarray(p, dtype=float), 1e-7, 1 - 1e-7)
        return np.log(p / (1 - p)).reshape(-1, 1)


# ---------- Head A ----------

def event_metrics(y, probs_a):
    """Per-bucket P/R/F1 + 11x11 confusion matrix, on rows that have an event-type label."""
    y = np.asarray(y)
    m = y != UNLABELED
    y, pred = y[m], np.asarray(probs_a)[m].argmax(1)
    labels = list(range(N_EVENT_CLASSES))
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    per = pd.DataFrame({"bucket": EVENT_BUCKETS, "precision": p, "recall": r, "f1": f, "support": s})
    per["note"] = ""
    per.loc[per.support < 30, "note"] = "UNRELIABLE: n<30, not statistically meaningful"
    cm = pd.DataFrame(confusion_matrix(y, pred, labels=labels),
                      index=[f"true_{b}" for b in EVENT_BUCKETS], columns=[f"pred_{b}" for b in EVENT_BUCKETS])
    summary = {"accuracy": float((pred == y).mean()),
               "macro_f1": float(f.mean()),
               "weighted_f1": float(np.average(f, weights=s)),
               "n": int(len(y))}
    return per, cm, summary


def sh_metrics(y, probs_a):
    """One-vs-rest P/R/F1 for SH."""
    y = np.asarray(y)
    m = y != UNLABELED
    yt, pred = (y[m] == SH).astype(int), (np.asarray(probs_a)[m].argmax(1) == SH).astype(int)
    p, r, f, _ = precision_recall_fscore_support(yt, pred, average="binary", zero_division=0)
    return {"n_true_SH": int(yt.sum()), "n_pred_SH": int(pred.sum()), "precision": float(p),
            "recall": float(r), "f1": float(f)}


# ---------- Guardrails ----------

def check_rare_bucket_collapse(per_bucket, name="", min_support=10):
    """Rare-bucket collapse: a bucket with real support but ~zero recall."""
    bad = per_bucket[(per_bucket.support >= min_support) & (per_bucket.recall < 0.05)]
    for _, r in bad.iterrows():
        warn(f"rare-bucket collapse [{name}]: {r.bucket} recall {r.recall:.3f} on n={r.support}")
    return bad.bucket.tolist()


def check_suspicious_performance(harm_val, event_val, name=""):
    """Leakage: near-perfect offline numbers usually mean leakage."""
    if harm_val["pr_auc"] > 0.98:
        warn(f"possible label leakage [{name}]: validation hurt PR-AUC {harm_val['pr_auc']:.3f}")
    if event_val["accuracy"] > 0.995:
        warn(f"possible label leakage [{name}]: validation event-type accuracy {event_val['accuracy']:.3f}")


def check_head_balance(history):
    """Loss imbalance: one head near chance while the other improves (history: list of per-epoch val dicts)."""
    if len(history) < 2:
        return
    last, first = history[-1], history[0]
    a_chance = last["event_macro_f1"] < 1.5 / N_EVENT_CLASSES
    b_chance = last["harm_pr_auc"] < 1.5 * last["harm_prevalence"]
    if a_chance and last["harm_pr_auc"] > first["harm_pr_auc"]:
        warn("loss imbalance: Head A near chance while Head B improves -- raise w_a")
    if b_chance and last["event_macro_f1"] > first["event_macro_f1"]:
        warn("loss imbalance: Head B near chance while Head A improves -- raise w_b")


def bucket_sensitivity(buckets, p_with, p_without, tol=0.005, name=""):
    """Event-type ceiling signature: per bucket, mean |p_hurt(with Head A input) - p_hurt(without)|.

    `p_without` is either the ablation model's p_hurt, or the full model's p_hurt with Head A's
    vector replaced by the train prior. Warns for every bucket whose predictions are ~identical.
    """
    buckets = np.asarray(buckets, dtype=object)
    d = np.abs(np.asarray(p_with) - np.asarray(p_without))
    rows = []
    for b in EVENT_BUCKETS:
        m = buckets == b
        if m.any():
            rows.append({"bucket": b, "n": int(m.sum()), "mean_abs_diff": float(d[m].mean()),
                         "max_abs_diff": float(d[m].max())})
    t = pd.DataFrame(rows)
    flat = t[t.mean_abs_diff < tol]
    for _, r in flat.iterrows():
        warn(f"event-type ceiling signature [{name}]: bucket {r.bucket} hurt probabilities ~identical with vs "
             f"without Head A input (mean |diff| {r.mean_abs_diff:.4f} < {tol}, n={r.n})")
    return t

In [ ]:
%%writefile train_encoder.py
"""Fine-tuning loop for the encoder multi-task model.

Used by the Colab notebook and by the local smoke test. Checkpoints every
`ckpt_every` optimizer steps to `ckpt_dir` (point this at Google Drive in Colab)
and resumes from the latest checkpoint if one exists.
"""
import json
import os
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

from evaluation import check_head_balance, event_metrics
from mtl import EncoderMultiTask, event_prior_from_labels, linear_warmup_decay, multitask_loss, predict
from pipeline import EVENT_BUCKETS, assert_no_excluded_columns
from sklearn.metrics import average_precision_score


class ReportDataset(Dataset):
    def __init__(self, df):
        # The model only ever sees `text`; hard-fail if anything excluded sneaks in.
        assert_no_excluded_columns(df.columns)
        self.text = df["text"].tolist()
        self.y_event = df["event_type_label"].to_numpy()
        self.y_harm = df["harm_label"].to_numpy()

    def __len__(self):
        return len(self.text)

    def __getitem__(self, i):
        return self.text[i], self.y_event[i], self.y_harm[i]


def make_collate(tokenizer, max_length):
    def collate(batch):
        text, ye, yh = zip(*batch)
        enc = tokenizer(list(text), truncation=True, max_length=max_length, padding=True, return_tensors="pt")
        inputs = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        return inputs, torch.as_tensor(np.array(ye)), torch.as_tensor(np.array(yh))
    return collate


def _ckpts(ckpt_dir):
    if not os.path.isdir(ckpt_dir):
        return []
    return sorted(f for f in os.listdir(ckpt_dir) if f.startswith("step_") and f.endswith(".pt"))


def _latest_ckpt(ckpt_dir):
    cks = _ckpts(ckpt_dir)
    return os.path.join(ckpt_dir, cks[-1]) if cks else None


def _save_ckpt(path, model, opt, sched, scaler, state):
    tmp = path + ".tmp"
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict() if scaler else None, "state": state}, tmp)
    os.replace(tmp, path)  # atomic, so a disconnect mid-save can't corrupt the latest checkpoint


def token_lengths(tok, texts, max_length):
    return np.array([len(x) for x in tok(list(texts), truncation=True, max_length=max_length)["input_ids"]])


def epoch_batches(lengths, batch_size, seed, bucket=True, chunk_batches=50):
    """Batch index lists for one epoch, reproducible from `seed` (so a resume replays the same order).

    With `bucket`, the shuffled rows are cut into chunks of `chunk_batches` batches and each chunk is
    sorted by token length, so a batch holds similarly sized reports and dynamic padding wastes little.
    Batch order is then shuffled again so training doesn't see a short-to-long curriculum.
    """
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(lengths))
    batches = []
    chunk = batch_size * chunk_batches if bucket else len(idx)
    for i in range(0, len(idx), chunk):
        c = idx[i:i + chunk]
        if bucket:
            c = c[np.argsort(lengths[c], kind="stable")]
        batches += [c[j:j + batch_size].tolist() for j in range(0, len(c), batch_size)]
    return [batches[k] for k in rng.permutation(len(batches))] if bucket else batches


def predict_df(model, tok, df, cfg, device, event_prior=None):
    """Predict in length-sorted order (minimal padding), return results in the original row order."""
    order = np.argsort(token_lengths(tok, df["text"], cfg["max_length"]), kind="stable")
    loader = DataLoader(ReportDataset(df.iloc[order]), batch_size=cfg["eval_batch_size"],
                        collate_fn=make_collate(tok, cfg["max_length"]))
    pred = predict(model, loader, device, event_prior)
    inv = np.empty_like(order)
    inv[order] = np.arange(len(order))
    return {k: v[inv] for k, v in pred.items()}


def val_summary(model, tok, val_df, cfg, device):
    """Same validation score as the baseline's early stopping: harm PR-AUC + event macro-F1."""
    pred = predict_df(model, tok, val_df, cfg, device)
    _, _, ev = event_metrics(val_df.event_type_label.to_numpy(), pred["probs_a"])
    y = val_df.harm_label.to_numpy()
    m = y >= 0
    return {"event_macro_f1": ev["macro_f1"], "event_accuracy": ev["accuracy"],
            "harm_pr_auc": float(average_precision_score(y[m], pred["p_hurt"][m])),
            "harm_prevalence": float(y[m].mean())}


def fit(train_df, val_df, cfg, device, log=print):
    """Train one model (full or ablation) per `cfg`. Returns (model, tokenizer, history).

    Validation-based early stopping (same score as the baseline: val harm PR-AUC + event macro-F1):
    evaluate every `eval_every` optimizer steps, stop after `patience` evaluations without
    improvement, cap at `max_epochs`, and restore the best weights at the end.
    """
    from transformers import AutoTokenizer

    torch.manual_seed(cfg["seed"])
    os.makedirs(cfg["ckpt_dir"], exist_ok=True)
    tok = AutoTokenizer.from_pretrained(cfg["model_name"])
    collate = make_collate(tok, cfg["max_length"])
    train_ds = ReportDataset(train_df)
    lengths = token_lengths(tok, train_df["text"], cfg["max_length"])

    model = EncoderMultiTask(cfg["model_name"], use_event_probs=cfg["use_event_probs"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
    n_batches = -(-len(train_ds) // cfg["batch_size"])
    steps_per_epoch = n_batches // cfg["grad_accum"]
    total = steps_per_epoch * cfg["max_epochs"]
    if cfg.get("max_steps"):
        total = min(total, cfg["max_steps"])
    sched = linear_warmup_decay(opt, int(0.06 * total), total)
    use_amp = device == "cuda" and cfg.get("fp16", True)
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    best_path = os.path.join(cfg["ckpt_dir"], "best_model.pt")

    identity = {"model_name": cfg["model_name"], "use_event_probs": cfg["use_event_probs"]}
    batching = {k: cfg[k] for k in ("batch_size", "grad_accum", "max_length", "seed")}
    batching["length_bucketing"] = cfg.get("length_bucketing", True)
    state = {"step": 0, "epoch": 0, "batch_in_epoch": 0, "history": [], "best_score": -1.0,
             "bad_evals": 0, "stopped": False, "identity": identity, "batching": batching}
    ck = _latest_ckpt(cfg["ckpt_dir"])
    if ck:
        blob = torch.load(ck, map_location=device, weights_only=False)
        saved = blob["state"]
        # Check compatibility before touching any weights, so a stale checkpoint gives a clear message.
        if saved.get("identity") != identity:
            raise RuntimeError(
                f"{cfg['ckpt_dir']} holds a checkpoint from a different model ({saved.get('identity') or 'an older run with no model record, e.g. Clinical-Longformer'}); "
                f"this run is {identity}. Point ckpt_dir at a fresh folder, or move that folder aside if you "
                f"no longer need it. Nothing was loaded.")
        if saved.get("batching") != batching:
            raise RuntimeError(f"Checkpoint {ck} was trained with {saved.get('batching')}, but this run uses "
                               f"{batching}; the saved position in the epoch would not line up. Set the same values "
                               f"(e.g. batch_size/grad_accum from outputs/batch_probe.json) or use a fresh ckpt_dir.")
        model.load_state_dict(blob["model"])
        opt.load_state_dict(blob["opt"])
        sched.load_state_dict(blob["sched"])
        if scaler and blob["scaler"]:
            scaler.load_state_dict(blob["scaler"])
        state = saved
        log(f"Resumed from {ck} at step {state['step']} (epoch {state['epoch']}, batch {state['batch_in_epoch']})")

    def save_ckpt():
        _save_ckpt(os.path.join(cfg["ckpt_dir"], f"step_{state['step']:07d}.pt"), model, opt, sched, scaler, state)
        for old in _ckpts(cfg["ckpt_dir"])[:-cfg["keep_ckpts"]]:
            os.remove(os.path.join(cfg["ckpt_dir"], old))

    def evaluate():
        vs = val_summary(model, tok, val_df, cfg, device)
        vs.update(step=state["step"], epoch=state["epoch"])
        state["history"].append(vs)
        log(f"validation @ step {state['step']}: {vs}")
        check_head_balance(state["history"])
        score = vs["harm_pr_auc"] + vs["event_macro_f1"]
        if score > state["best_score"]:
            state["best_score"], state["bad_evals"] = score, 0
            torch.save(model.state_dict(), best_path)
            log(f"  new best (val harm PR-AUC + event macro-F1 = {score:.4f}) -> best_model.pt")
        else:
            state["bad_evals"] += 1
            log(f"  no improvement ({state['bad_evals']}/{cfg['patience']})")
            if state["bad_evals"] >= cfg["patience"]:
                state["stopped"] = True
                log(f"Early stopping at step {state['step']}: best score {state['best_score']:.4f}")
        model.train()

    json.dump(cfg, open(os.path.join(cfg["ckpt_dir"], "config.json"), "w"), indent=2)
    log(f"{len(train_ds)} train rows, {steps_per_epoch} optimizer steps/epoch, cap {total} steps "
        f"({cfg['max_epochs']} epochs), eval every {cfg['eval_every']} steps, patience {cfg['patience']}, "
        f"batch {cfg['batch_size']} x accum {cfg['grad_accum']}, fp16={use_amp}, "
        f"length bucketing={cfg.get('length_bucketing', True)}")
    t0, start_step = time.time(), state["step"]
    while not state["stopped"] and state["step"] < total and state["epoch"] < cfg["max_epochs"]:
        model.train()
        batches = epoch_batches(lengths, cfg["batch_size"], cfg["seed"] + state["epoch"],
                                bucket=cfg.get("length_bucketing", True))
        start_b = state["batch_in_epoch"]  # resume mid-epoch: skip batches already trained on
        loader = DataLoader(train_ds, batch_sampler=batches[start_b:], collate_fn=collate)
        opt.zero_grad(set_to_none=True)  # don't carry a partial accumulation across epochs/resumes
        for b, (inputs, ye, yh) in enumerate(loader, start=start_b):
            inputs = {k: v.to(device) for k, v in inputs.items()}
            ye, yh = ye.to(device), yh.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                la, lb = model(**inputs)
            loss, ce, bce = multitask_loss(la.float(), lb.float(), ye, yh, cfg["w_a"], cfg["w_b"])
            loss = loss / cfg["grad_accum"]
            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            state["batch_in_epoch"] = b + 1
            if (b + 1) % cfg["grad_accum"]:
                continue
            if scaler:
                scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if scaler:
                scaler.step(opt)
                scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True)
            sched.step()
            state["step"] += 1
            if device == "mps" and state["step"] % 10 == 0:
                torch.mps.empty_cache()  # MPS caches a buffer per batch shape; dynamic padding makes many
            if state["step"] % cfg["log_every"] == 0:
                rate = (state["step"] - start_step) / max(1e-9, time.time() - t0)
                eta_h = (total - state["step"]) / max(rate, 1e-9) / 3600
                log(f"step {state['step']}/{total} loss {loss.item() * cfg['grad_accum']:.4f} "
                    f"(CE {ce.item():.4f}, BCE {bce.item():.4f}) {rate:.2f} steps/s, "
                    f"ETA to the {total}-step cap {eta_h:.2f} h (early stopping may end sooner)")
            if state["step"] % cfg["eval_every"] == 0:
                evaluate()
            if state["step"] % cfg["ckpt_every"] == 0 or state["stopped"]:
                save_ckpt()
            if state["stopped"] or state["step"] >= total:
                break
        else:
            state["epoch"] += 1
            state["batch_in_epoch"] = 0
            save_ckpt()
            continue
        break

    if not state["history"] or state["history"][-1]["step"] != state["step"]:
        evaluate()  # score the final weights too, so the best checkpoint considers them
        save_ckpt()
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    return model, tok, state["history"]


def probe_batch_size(model_name, max_length, device, candidates=(8, 16, 32, 64, 128), headroom=0.85, log=print):
    """Largest batch size whose worst case (every row at max_length) survives forward + backward +
    optimizer step, using the same mixed-precision setup as training. Returns (best, report)."""
    report, best = [], None
    use_amp = device == "cuda"
    if device == "cuda":
        total_mem = torch.cuda.get_device_properties(0).total_memory
    elif device == "mps":
        total_mem = torch.mps.recommended_max_memory()
    else:
        return None, [{"note": "no GPU"}]
    for bs in candidates:
        model = opt = None
        try:
            model = EncoderMultiTask(model_name, use_event_probs=True).to(device)
            opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
            scaler = torch.amp.GradScaler("cuda") if use_amp else None
            if device == "cuda":
                torch.cuda.reset_peak_memory_stats()
            ids = torch.randint(1000, 20000, (bs, max_length), device=device)
            mask = torch.ones_like(ids)
            for _ in range(2):  # second step includes the allocated optimizer state
                with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    la, lb = model(ids, mask)
                loss = la.float().logsumexp(-1).mean() + lb.float().mean()
                if scaler:
                    scaler.scale(loss).backward()
                    scaler.step(opt)
                    scaler.update()
                else:
                    loss.backward()
                    opt.step()
                opt.zero_grad(set_to_none=True)
            if device == "cuda":
                torch.cuda.synchronize()
                peak = torch.cuda.max_memory_reserved()
            else:
                torch.mps.synchronize()
                peak = torch.mps.driver_allocated_memory()  # current driver allocation (MPS has no peak counter)
            ok = peak < headroom * total_mem
            report.append({"batch_size": bs, "peak_gb": round(peak / 2 ** 30, 2),
                           "device_gb": round(total_mem / 2 ** 30, 2), "fits_with_headroom": ok})
            log(f"batch {bs} x {max_length} tokens: peak {peak / 2 ** 30:.2f} GB of {total_mem / 2 ** 30:.2f} GB"
                f" -> {'OK' if ok else 'too close to the limit'}")
            if not ok:
                break
            best = bs
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            report.append({"batch_size": bs, "oom": True})
            log(f"batch {bs} x {max_length} tokens: OOM")
            break
        finally:
            del model, opt
            if device == "cuda":
                torch.cuda.empty_cache()
            elif device == "mps":
                torch.mps.empty_cache()
    return best, report


def predictions_frame(df, pred):
    """Uniform prediction file format consumed by make_report.py."""
    out = df[["event_no", "split", "event_bucket", "event_type_label", "harm_label"]].reset_index(drop=True).copy()
    out["p_hurt"] = pred["p_hurt"]
    if "p_hurt_prior" in pred:
        out["p_hurt_prior"] = pred["p_hurt_prior"]
    for i, b in enumerate(EVENT_BUCKETS):
        out[f"p_event_{b}"] = pred["probs_a"][:, i]
    return out


def train_prior(train_df):
    return event_prior_from_labels(train_df.event_type_label.to_numpy())

In [ ]:
%%writefile make_report.py
"""Step 6: evaluation report from saved prediction files.

Reads outputs/<variant>/predictions_{validation,test}*.parquet for every variant that
exists (baseline, baseline_ablation, main, main_ablation) and writes
reports/evaluation_report.md plus CSV/PNG/JSON artifacts. Variants that haven't been
run yet (the Colab-only main model) are reported as "not run" -- nothing is estimated.

Usage: python src/make_report.py
"""
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd

from evaluation import (PlattCalibrator, bucket_sensitivity, calibration_table, check_rare_bucket_collapse,
                        check_suspicious_performance, choose_threshold, event_metrics, harm_metrics, sh_metrics)
from pipeline import EVENT_BUCKETS, UNLABELED

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
OUT = os.path.join(ROOT, "outputs")
REP = os.path.join(ROOT, "reports")
PCOLS = [f"p_event_{b}" for b in EVENT_BUCKETS]
VARIANTS = {
    "baseline": "Baseline: MiniLM (frozen) + MLP heads, Head B gets Head A probs",
    "baseline_ablation": "Baseline ablation: MiniLM + MLP, Head B text-only",
    "main": "Main: Bio_ClinicalBERT fine-tuned, Head B gets Head A probs",
    "main_ablation": "Main ablation: Bio_ClinicalBERT, Head B text-only",
}


def load(variant, split, seed=None):
    d = os.path.join(OUT, variant)
    name = f"predictions_{split}" + (f"_seed{seed}" if seed is not None else "") + ".parquet"
    p = os.path.join(d, name)
    return pd.read_parquet(p) if os.path.exists(p) else None


def primary(variant, split):
    """Seed-0 run for multi-seed variants, else the single run."""
    return load(variant, split, 0) if load(variant, split, 0) is not None else load(variant, split)


def harm_rows(df):
    return df[df.harm_label != UNLABELED]


def evaluate_variant(v):
    val, test = primary(v, "validation"), primary(v, "test")
    if val is None or test is None:
        return None
    hv, ht = harm_rows(val), harm_rows(test)
    thr = choose_threshold(hv.harm_label, hv.p_hurt)
    res = {"threshold": thr,
           "val_harm": harm_metrics(hv.harm_label, hv.p_hurt, thr["threshold"]),
           "test_harm": harm_metrics(ht.harm_label, ht.p_hurt, thr["threshold"])}
    # Calibration: raw, then Platt fit on validation, both checked on test.
    cal_raw, ece_raw = calibration_table(ht.harm_label, ht.p_hurt)
    platt = PlattCalibrator().fit(hv.harm_label.to_numpy(), hv.p_hurt.to_numpy())
    p_cal = platt.transform(ht.p_hurt)
    cal_cal, ece_cal = calibration_table(ht.harm_label, p_cal)
    thr_cal = float(platt.transform([thr["threshold"]])[0])
    res["calibration"] = {"ece_raw": ece_raw, "ece_platt": ece_cal, "threshold_on_calibrated_scale": thr_cal,
                          "table_raw": cal_raw, "table_platt": cal_cal,
                          "test_recall_at_calibrated_threshold": harm_metrics(ht.harm_label, p_cal, thr_cal)["recall"]}
    per, cm, summ = event_metrics(test.event_type_label, test[PCOLS].to_numpy())
    _, _, summ_v = event_metrics(val.event_type_label, val[PCOLS].to_numpy())
    res.update(event_per_bucket=per, event_cm=cm, event_summary=summ, event_summary_val=summ_v)
    res["sh_test"] = sh_metrics(test.event_type_label, test[PCOLS].to_numpy())
    res["sh_val"] = sh_metrics(val.event_type_label, val[PCOLS].to_numpy())
    oof = load(v, "train_oof", 0)
    tr_in = primary(v, "train")
    if oof is not None:
        both = pd.concat([oof, val])
        res["sh_trainval"] = {**sh_metrics(both.event_type_label, both[PCOLS].to_numpy()),
                              "train_source": "5-fold out-of-fold predictions on TRAIN (honest)"}
    elif tr_in is not None:
        both = pd.concat([tr_in, val])
        res["sh_trainval"] = {**sh_metrics(both.event_type_label, both[PCOLS].to_numpy()),
                              "train_source": "IN-SAMPLE predictions on TRAIN (optimistic: model trained on these rows)"}
    res["guardrail_rare_collapse"] = check_rare_bucket_collapse(per, v)
    check_suspicious_performance(res["val_harm"], summ_v, v)
    res["_test"] = test
    return res


def seed_tables(v):
    """Per-seed hurt-class metrics and per-bucket event metrics.

    The threshold is re-chosen on each seed's own validation predictions (nothing reused).
    """
    rows, buckets = [], []
    for seed in range(20):
        val, test = load(v, "validation", seed), load(v, "test", seed)
        if val is None:
            continue
        hv, ht = harm_rows(val), harm_rows(test)
        thr = choose_threshold(hv.harm_label, hv.p_hurt)
        m = harm_metrics(ht.harm_label, ht.p_hurt, thr["threshold"])
        per, _, ev = event_metrics(test.event_type_label, test[PCOLS].to_numpy())
        rows.append({"variant": v, "seed": seed, "threshold": thr["threshold"], "val_recall": thr["val_recall"],
                     "test_recall": m["recall"], "test_precision": m["precision"], "test_f1": m["f1"],
                     "test_pr_auc": m["pr_auc"], "test_event_accuracy": ev["accuracy"],
                     "test_event_macro_f1": ev["macro_f1"]})
        buckets.append(per.assign(variant=v, seed=seed))
    return pd.DataFrame(rows), (pd.concat(buckets, ignore_index=True) if buckets else pd.DataFrame())


def train_configs():
    """Training config + best epochs per baseline variant, to confirm the comparison is like-for-like."""
    out = {}
    runs = os.path.join(OUT, "baseline_seed_runs.csv")
    runs = pd.read_csv(runs) if os.path.exists(runs) else pd.DataFrame()
    for v in ("baseline", "baseline_ablation"):
        p = os.path.join(OUT, v, "train_log_seed0.json")
        if os.path.exists(p):
            cfg = json.load(open(p))["config"]
            eps = runs[runs.variant == v].best_epoch.tolist() if len(runs) else []
            out[v] = {"config": cfg, "best_epochs": eps}
    return out


def pm(series):
    return f"{series.mean():.3f} ± {series.std():.3f}"


def save_cm_png(cm, title, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    rn = cm.div(cm.sum(1).replace(0, 1), axis=0)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(rn.values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(11), EVENT_BUCKETS)
    ax.set_yticks(range(11), EVENT_BUCKETS)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    for i in range(11):
        for j in range(11):
            n = cm.values[i, j]
            if n:
                ax.text(j, i, str(n), ha="center", va="center", fontsize=7,
                        color="white" if rn.values[i, j] > 0.5 else "black")
    ax.set_title(title + "\n(color = row-normalized recall; numbers = counts)")
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def fmt(x, pct=False):
    return f"{x:.1%}" if pct else f"{x:.3f}"


def md_table(df, floatfmt="{:.3f}"):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(floatfmt.format(v) if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def main():
    os.makedirs(REP, exist_ok=True)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        results = {v: evaluate_variant(v) for v in VARIANTS}
    guardrail_msgs = [str(w.message) for w in caught if w.category.__name__ == "GuardrailWarning"]
    ran = [v for v, r in results.items() if r]

    # ----- Guardrail: per-bucket sensitivity of Head B to Head A's input -----
    sens = {}
    for full, abl in (("baseline", "baseline_ablation"), ("main", "main_ablation")):
        if results.get(full):
            t = harm_rows(results[full]["_test"])
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                if "p_hurt_prior" in t:
                    sens[f"{full}: with vs Head A replaced by train prior"] = bucket_sensitivity(
                        t.event_bucket, t.p_hurt, t.p_hurt_prior, name=f"{full}, Head A -> prior")
                if results.get(abl):
                    ta = harm_rows(results[abl]["_test"]).set_index("event_no").loc[t.event_no]
                    sens[f"{full} vs {abl} (separately trained)"] = bucket_sensitivity(
                        t.event_bucket, t.p_hurt.to_numpy(), ta.p_hurt.to_numpy(), name=f"{full} vs {abl}")
            guardrail_msgs += [str(x.message) for x in w if x.category.__name__ == "GuardrailWarning"]

    st = [seed_tables(v) for v in ("baseline", "baseline_ablation")]
    seeds = pd.concat([a for a, _ in st], ignore_index=True)
    seed_buckets = pd.concat([b for _, b in st], ignore_index=True)
    cfgs = train_configs()

    # ----- artifacts -----
    metrics_json = {}
    for v in ran:
        r = results[v]
        r["event_per_bucket"].to_csv(os.path.join(REP, f"per_bucket_{v}.csv"), index=False)
        r["event_cm"].to_csv(os.path.join(REP, f"confusion_matrix_{v}.csv"))
        save_cm_png(r["event_cm"], f"{VARIANTS[v]} - test set", os.path.join(REP, f"confusion_matrix_{v}.png"))
        r["calibration"]["table_raw"].to_csv(os.path.join(REP, f"calibration_{v}_raw.csv"), index=False)
        r["calibration"]["table_platt"].to_csv(os.path.join(REP, f"calibration_{v}_platt.csv"), index=False)
        t = harm_rows(r["_test"])
        t[["event_no", "p_hurt"]].assign(pred_hurt=(t.p_hurt >= r["threshold"]["threshold"]).astype(int)).to_csv(
            os.path.join(REP, f"test_predictions_{v}.csv"), index=False)
        metrics_json[v] = {k: r[k] for k in ("threshold", "val_harm", "test_harm", "event_summary",
                                             "event_summary_val", "sh_test", "sh_val") }
        metrics_json[v]["sh_trainval"] = r.get("sh_trainval")
        metrics_json[v]["calibration"] = {k: r["calibration"][k] for k in
                                          ("ece_raw", "ece_platt", "threshold_on_calibrated_scale",
                                           "test_recall_at_calibrated_threshold")}
    for name, t in sens.items():
        t.assign(comparison=name).to_csv(os.path.join(REP, "bucket_sensitivity.csv"),
                                         mode="a" if name != list(sens)[0] else "w",
                                         header=name == list(sens)[0], index=False)
    if len(seeds):
        seeds.to_csv(os.path.join(REP, "baseline_seed_runs_test.csv"), index=False)
        seed_buckets.to_csv(os.path.join(REP, "baseline_seed_runs_per_bucket_test.csv"), index=False)
    json.dump({"metrics": metrics_json, "guardrail_warnings": guardrail_msgs},
              open(os.path.join(REP, "metrics.json"), "w"), indent=2, default=float)

    # ----- markdown report -----
    L = ["# Evaluation report — harm prediction via 11 event-type buckets", "",
         "Generated by `src/make_report.py` from saved prediction files. All metrics are on the **test set** "
         "(Oct 2024, natural prevalence) unless labeled otherwise; thresholds were chosen on **validation** only.", ""]
    L += ["## Which models ran", "", "| Variant | Status |", "|---|---|"]
    for v, desc in VARIANTS.items():
        L.append(f"| {desc} | {'evaluated' if results[v] else '**not run yet** (Colab-only; run the notebook, copy predictions into `outputs/' + v + '/`, re-run this script)'} |")
    L.append("")

    L += ["## Threshold selection", ""]
    for v in ran:
        th = results[v]["threshold"]
        L.append(f"- **{v}**: threshold **{th['threshold']:.4f}** on P(hurt). Rule: {th['rule']}. "
                 f"On validation that gives recall {fmt(th['val_recall'], True)}, precision {fmt(th['val_precision'], True)}.")
    L.append("")

    L += ["## Head B: hurt-class metrics, test set", ""]
    rows = []
    for v in ran:
        m = results[v]["test_harm"]
        rows.append({"model": v, "recall": fmt(m["recall"], True), "precision": fmt(m["precision"], True),
                     "F1": fmt(m["f1"]), "PR-AUC": fmt(m["pr_auc"]), "ROC-AUC": fmt(m["roc_auc"]),
                     "TP": m["tp"], "FN": m["fn"], "FP": m["fp"], "flagged": fmt(m["flagged_frac"], True)})
    if rows:
        L.append(md_table(pd.DataFrame(rows)))
        m0 = results[ran[0]]["test_harm"]
        L += ["", f"Test hurt prevalence {fmt(m0['prevalence'], True)} (n={m0['n']}, {m0['n_hurt']} hurt). "
              f"PR-AUC of a random model = prevalence. A model that always says \"not hurt\" scores "
              f"{fmt(m0['always_not_hurt_accuracy'], True)} accuracy, which is why accuracy isn't reported here.", ""]

    L += ["## Ablation: does Head A's event-type vector help Head B?", ""]
    for full, abl in (("baseline", "baseline_ablation"), ("main", "main_ablation")):
        if results.get(full) and results.get(abl):
            a, b = results[full]["test_harm"], results[abl]["test_harm"]
            L.append(f"**{full} vs {abl}** (seed 0): PR-AUC {fmt(a['pr_auc'])} vs {fmt(b['pr_auc'])} "
                     f"(Δ {a['pr_auc'] - b['pr_auc']:+.3f}); recall {fmt(a['recall'], True)} vs {fmt(b['recall'], True)}; "
                     f"precision {fmt(a['precision'], True)} vs {fmt(b['precision'], True)}.")
        else:
            L.append(f"**{full} vs {abl}**: not run yet.")
    if len(seeds):
        n_seeds = seeds.groupby("variant").seed.nunique().to_dict()
        L += ["", f"### Baseline across seeds: mean ± sd, test set (seeds per variant: {n_seeds})", "",
              "Each seed's threshold was re-chosen on that seed's own validation predictions, from these "
              "retrained models. No threshold is carried over from the earlier undertrained runs.", ""]
        cols = ["threshold", "val_recall", "test_recall", "test_precision", "test_f1", "test_pr_auc",
                "test_event_accuracy", "test_event_macro_f1"]
        tab = seeds.groupby("variant")[cols].agg(pm).reset_index()
        L.append(md_table(tab))
        f = seeds[seeds.variant == "baseline"].set_index("seed")
        g = seeds[seeds.variant == "baseline_ablation"].set_index("seed")
        L += ["", "Paired difference, full − ablation (same seed):", ""]
        diff = pd.DataFrame({c: [pm((f[c] - g[c]).dropna())] for c in
                             ("test_recall", "test_precision", "test_f1", "test_pr_auc", "test_event_macro_f1")})
        L.append(md_table(diff))
        d = (f.test_pr_auc - g.test_pr_auc).dropna()
        noise = max(seeds.groupby("variant").test_pr_auc.std().max(), 1e-9)
        wins = int((d > 0).sum())
        verdict = ("**meaningful**" if abs(d.mean()) > 2 * noise and abs(d.mean()) > 0.01 else
                   "**not meaningful** (within 2× seed-to-seed noise, or under 0.01 PR-AUC)")
        L.append(f"\nVerdict for the baseline: the PR-AUC gap of {d.mean():+.4f} is {verdict}. "
                 f"Largest per-variant seed-to-seed PR-AUC sd = {noise:.4f}; the full model beat the ablation "
                 f"on {wins}/{len(d)} seeds.")
    if cfgs:
        same = len({json.dumps(c["config"], sort_keys=True) for c in cfgs.values()}) == 1
        L += ["", "### Fairness check: identical training settings", ""]
        for v, c in cfgs.items():
            cap = c["config"]["max_epochs"]
            L.append(f"- **{v}**: {c['config']} — best epochs per seed {c['best_epochs']} "
                     f"({'all below' if c['best_epochs'] and max(c['best_epochs']) < cap - 1 else 'NOT all below'} the {cap}-epoch cap).")
        L.append(f"- Configs identical: **{'yes' if same else 'NO'}**. Same frozen embeddings, same seeds (0–4), same "
                 "train/validation/test rows, same early-stopping rule (validation harm PR-AUC + event macro-F1). "
                 "The only difference is whether Head B receives Head A's 11-way probability vector.")
    L.append("")

    if len(seed_buckets):
        L += ["### Head A per bucket across seeds: mean ± sd, test set", ""]
        agg = seed_buckets.groupby(["bucket", "variant"]).agg(
            precision=("precision", pm), recall=("recall", pm), f1=("f1", pm), support=("support", "first")).reset_index()
        agg["bucket"] = pd.Categorical(agg.bucket, EVENT_BUCKETS, ordered=True)
        agg = agg.sort_values(["bucket", "variant"])
        agg["note"] = np.where(agg.bucket == "SH", "NOT STATISTICALLY MEANINGFUL (n=1 in test)", "")
        L += [md_table(agg), ""]

    L += ["## Head A: event type, test set", ""]
    for v in ran:
        r = results[v]
        s = r["event_summary"]
        L += [f"### {v}", "", f"Accuracy {fmt(s['accuracy'], True)}, macro-F1 {fmt(s['macro_f1'])}, "
              f"weighted-F1 {fmt(s['weighted_f1'])} on n={s['n']} test rows with an event-type code "
              f"(the 7 uncoded test rows are excluded from Head A).", ""]
        per = r["event_per_bucket"].copy()
        per.loc[per.bucket == "SH", "note"] = "NOT STATISTICALLY MEANINGFUL: n=1 in test; see supplementary SH below"
        L.append(md_table(per))
        L += ["", f"Confusion matrix: `reports/confusion_matrix_{v}.csv` / `.png`.", ""]
        sh = r["sh_test"]
        L.append(f"**SH, test (n={sh['n_true_SH']}, NOT statistically meaningful):** precision {fmt(sh['precision'])}, "
                 f"recall {fmt(sh['recall'])}, F1 {fmt(sh['f1'])} ({sh['n_pred_SH']} test rows predicted SH).")
        if r.get("sh_trainval"):
            st = r["sh_trainval"]
            L.append(f"**SH, supplementary, train+val combined (n={st['n_true_SH']}):** precision {fmt(st['precision'])}, "
                     f"recall {fmt(st['recall'])}, F1 {fmt(st['f1'])}. Train part: {st['train_source']}.")
        sv = r["sh_val"]
        L += [f"SH, validation only (n={sv['n_true_SH']}): precision {fmt(sv['precision'])}, recall {fmt(sv['recall'])}, "
              f"F1 {fmt(sv['f1'])}.", ""]

    L += ["## Calibration check", "",
          "Raw probabilities come from a model trained at 15.1% hurt prevalence and are scored at ~4.8%, so they "
          "are expected to over-predict. Platt scaling (logistic on the logit) is fit on validation and checked on "
          "test. It is monotonic, so it doesn't change ranking, PR-AUC, or the recall/precision of the "
          "validation-chosen operating point; it only makes the probabilities readable as risks.", ""]
    for v in ran:
        c = results[v]["calibration"]
        L += [f"### {v}", "", f"ECE raw **{c['ece_raw']:.4f}** → after Platt **{c['ece_platt']:.4f}**. "
              f"Threshold {results[v]['threshold']['threshold']:.4f} raw = {c['threshold_on_calibrated_scale']:.4f} calibrated; "
              f"test recall at it: {fmt(c['test_recall_at_calibrated_threshold'], True)}.", ""]
        tr, tc = c["table_raw"], c["table_platt"]
        comb = pd.DataFrame({"bin": tr.bin, "n (raw)": tr.n, "mean pred (raw)": tr.mean_predicted,
                             "observed (raw)": tr.observed_hurt_rate, "n (Platt)": tc.n,
                             "mean pred (Platt)": tc.mean_predicted, "observed (Platt)": tc.observed_hurt_rate})
        L += [md_table(comb.fillna(np.nan)).replace("nan", "–"), ""]

    L += ["## Guardrails", ""]
    for name, t in sens.items():
        L += [f"**Per-bucket sensitivity to Head A, {name}** (mean |ΔP(hurt)| on test rows with a harm score; warn if < 0.005):", "",
              md_table(t, "{:.4f}"), ""]
    L += ["Guardrail warnings raised while building this report:", ""]
    L += [f"- {m}" for m in guardrail_msgs] or ["- none"]
    L += ["", "Hard checks enforced in code before any training (all passed): split sizes 60k/10k/10k; Event No. "
          "split agrees with Event Date; hurt prevalence within expected bands (train 13.5–16.5%, val 3.0–5.5%, "
          "test 3.5–6.5%); exactly 11 event-type labels in each Head A set; no excluded or label-source "
          "column in the model's features (`LeakageError`).", ""]

    L += ["## Comparison against the partner's model", "",
          "Not done: the partner's predictions aren't available here. Per-row test predictions for each model are in "
          "`reports/test_predictions_<variant>.csv` (Event No., P(hurt), 0/1 at the validation-chosen threshold), "
          "ready to join to the partner's output on `Event No.`. Before comparing, confirm the partner's 97% figure "
          "used these test rows, the A–D/E–I cut, and excluded rows with no harm score.", ""]

    open(os.path.join(REP, "evaluation_report.md"), "w").write("\n".join(L))
    print(f"wrote {os.path.join(REP, 'evaluation_report.md')} (variants: {ran})")


if __name__ == "__main__":
    sys.exit(main())

## 5. Data pipeline + hard checks

This fails loudly if the split, prevalence, 11-class set, or exclusion list is violated.

In [ ]:
import sys, json
sys.path.insert(0, '.')
import pandas as pd
from pipeline import load_dataset, validate_full_dataset, check_head_a_classes, assert_no_excluded_columns, SPLITS

df = load_dataset(CSV_PATH)
checks = validate_full_dataset(df)
print(json.dumps({k: v for k, v in checks.items()}, indent=2))
train_df, val_df, test_df = (df[df.split == s].reset_index(drop=True) for s in SPLITS)
for name, d in (('train', train_df), ('validation', val_df), ('test', test_df)):
    check_head_a_classes(d, name)
    assert_no_excluded_columns(d.columns)
os.makedirs(os.path.join(OUT_DIR, 'processed'), exist_ok=True)
df.to_parquet(os.path.join(OUT_DIR, 'processed', 'dataset.parquet'), index=False)
print(train_df.text.iloc[0][:300])

## 5b. Batch-size probe

This tries the worst case (every row at `max_length`) with a forward pass, backward pass and optimizer step in fp16, at batch 8, 16, 32, 64 and 128. It keeps the largest batch that stays under 85% of GPU memory, then sets `grad_accum` so each optimizer step still sees 32 rows. If 64 or 128 fits and you want a bigger effective batch, raise `EFFECTIVE_BATCH` (and consider scaling `lr`).

In [ ]:
import torch
from train_encoder import probe_batch_size
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'No GPU: switch the runtime to T4'
best, probe_report = probe_batch_size(BASE_CFG['model_name'], BASE_CFG['max_length'], device)
assert best, 'Even batch 8 does not fit at this max_length'
bs = min(best, EFFECTIVE_BATCH)
for cfg in CONFIGS.values():
    cfg.update(batch_size=bs, grad_accum=EFFECTIVE_BATCH // bs)
json.dump(probe_report, open(os.path.join(OUT_DIR, 'batch_probe.json'), 'w'), indent=2)
print(f"Largest batch that fits (worst case): {best}. Training with batch {bs} x grad_accum {EFFECTIVE_BATCH // bs} = {EFFECTIVE_BATCH} rows per optimizer step.")

## 5c. Speed check: ~50 optimizer steps before committing

This runs the real training loop and config for 50 steps into a throwaway folder (nothing is kept), then prints steps/s and the projected time. Continue to step 6 only if the ETA is acceptable. Early stopping usually ends training before the `max_epochs` cap, so the cap figure is an upper bound.

In [ ]:
import re, shutil, tempfile
from train_encoder import fit
speed_dir = tempfile.mkdtemp(dir='/content')  # throwaway timing run, deliberately not on Drive
speed_cfg = dict(CONFIGS['main'], max_steps=50, log_every=10, eval_every=10**9, ckpt_every=10**9, ckpt_dir=speed_dir)
speed_logs = []
fit(train_df, val_df.sample(512, random_state=0), speed_cfg, device, log=lambda m: (speed_logs.append(m), print(m)))
shutil.rmtree(speed_dir)
rate = [float(x) for l in speed_logs for x in re.findall(r'([\d.]+) steps/s', l)][-1]
spe = (-(-len(train_df) // CONFIGS['main']['batch_size'])) // CONFIGS['main']['grad_accum']
print(f"\n{rate:.2f} optimizer steps/s ({EFFECTIVE_BATCH} rows/step) -> {spe / rate / 3600:.2f} h per epoch; "
      f"upper bound {BASE_CFG['max_epochs'] * spe / rate / 3600:.2f} h per model at the {BASE_CFG['max_epochs']}-epoch cap "
      f"(x2 for main + ablation). Early stopping typically ends sooner.")

## 6. Train the main model (Head B gets Head A's soft probabilities)

This resumes automatically from the latest Drive checkpoint. It validates every `eval_every` steps and stops early after `patience` checks without improvement. If the session disconnects, re-run steps 1–5b and then this cell: the probe gives the same batch size, and training resumes where it stopped.

In [ ]:
from train_encoder import fit, predict_df, predictions_frame, train_prior

def run(variant):
    cfg = CONFIGS[variant]
    model, tok, history = fit(train_df, val_df, cfg, device)
    vdir = os.path.join(OUT_DIR, variant)
    os.makedirs(vdir, exist_ok=True)
    prior = train_prior(train_df) if cfg['use_event_probs'] else None
    for split, d in (('validation', val_df), ('test', test_df), ('train', train_df)):
        pred = predict_df(model, tok, d, cfg, device, event_prior=prior)
        predictions_frame(d, pred).to_parquet(os.path.join(vdir, f'predictions_{split}.parquet'))
    json.dump({'config': cfg, 'history': history}, open(os.path.join(vdir, 'train_log.json'), 'w'), indent=2)
    del model
    torch.cuda.empty_cache()
    return history

history_main = run('main')
history_main

## 7. Train the ablation (Head B text-only)

In [ ]:
history_ablation = run('main_ablation')
history_ablation

## 8. Evaluation report

This runs the same `make_report.py` as the local project, writing to `DRIVE_DIR/reports/`. It includes the validation-chosen threshold, hurt-class metrics for main vs ablation, the per-bucket confusion matrix, calibration, SH flagged as n=1 plus the supplementary train+val number, and the per-bucket event-type-reliance guardrail.

To fold the local baseline numbers into the same report: copy `outputs/baseline/` and `outputs/baseline_ablation/` from the local project into `DRIVE_DIR/outputs/` before running this cell. Or copy `DRIVE_DIR/outputs/main*` back to the local project and run `python src/make_report.py` there.

In [ ]:
import make_report
make_report.OUT = OUT_DIR
make_report.REP = os.path.join(DRIVE_DIR, 'reports')
make_report.main()
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(DRIVE_DIR, 'reports', 'evaluation_report.md')).read()))